# autocrop 2.0.0 visual examples

These examples use the 2.0.0 result API. For older calls, see the [migration FAQ](https://leblancfg.com/autocrop/faq/).
Run with Python 3.10+ and install Matplotlib and Jupyter alongside autocrop. Outputs are intentionally cleared: rerun the cells to see crops from your installed detector, rather than saved v1 images.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / 'autocrop').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'tests' / 'data'

from autocrop import Cropper, __version__

print('autocrop', __version__)
loc = str(DATA / 'macbeth.jpg')
plt.imshow(plt.imread(loc))


In [ ]:
# CropResult contains the image and diagnostics, including on a no-face attempt.
cropper = Cropper()
result = cropper.crop(loc)
if result.image is not None:
    display(Image.fromarray(result.image))
else:
    print(result.diagnostics.error)


In [ ]:
# Matplotlib also takes the checked image array, not the result record.
def show_crop(cropper, source):
    result = cropper.crop(source)
    if result.image is not None:
        plt.imshow(result.image)
        plt.show()
    else:
        print(f'No crop for {source}: {result.diagnostics.error}')
    return result

result = show_crop(cropper, loc)


In [ ]:
%%timeit
# Measure this environment; old Haar benchmarks do not describe YuNet.
result = cropper.crop(loc)


In [ ]:
# A larger face_percent requests a tighter crop.
result = show_crop(Cropper(face_percent=100), loc)


In [ ]:
result = show_crop(Cropper(height=700), loc)


In [ ]:
# Check the image field, not `if result` or `result is not None`.
noise = str(DATA / 'noise.png')
result = cropper.crop(noise)
assert result.image is None
print(result.diagnostics.error)


In [ ]:
faces = sorted(str(path) for path in DATA.iterdir() if path.suffix.lower() in {'.jpg', '.png'})
print(f'{len(faces)} images to test with.')

def plot_test_images(faces, cropper):
    for face in faces:
        try:
            show_crop(cropper, face)
        except OSError as error:
            # Unreadable files raise; they are not no-face results.
            print(f'Could not read {face}: {error}')

plot_test_images(faces, Cropper(width=650, height=500, face_percent=40))


In [ ]:
plot_test_images(faces, Cropper(height=650, width=500, face_percent=40))


## Crop coordinates and alignment

Read public diagnostics instead of calling the private `_crop_positions` helper. Face boxes use the EXIF-oriented input coordinates. If alignment rotates the input, the crop rectangle uses the aligned coordinate space. See the [API reference](https://leblancfg.com/autocrop/api/#alignment).


In [ ]:
result = show_crop(Cropper(align=True), loc)
print('Crop rectangle:', result.diagnostics.crop_rectangle)
print('Coordinate space:', result.diagnostics.crop_coordinate_space)
print('Alignment:', result.diagnostics.alignment)
